In [22]:
# these are the libraries that we need for our project
import pandas as pd

# Collection

I obtain data and inspect it.

In [23]:
data_path = "data/Pan-India_Bus_Routes.csv"

In [24]:
# ensure that all rows of a dataframe are displayed in the output. This will be useful for the value_counts() method
pd.set_option('display.max_rows', None)

bus_df = pd.read_csv(data_path, sep=",")

There is 35667 entries in this dataframe. Each refers to a bus route from one city to another, rather than to the bus itself. Therefore, we do not know how many buses are there, and how many of them have traits, such as A/C or Sleeper. However, we know for sure one thing. That those traits apply to the trip undertaken on the bus. And therefore, the business insights that we extract from the data can be applied to the bus trips, as they, themselves, are unique, and are what we are looking to improve based on insights from data.

In [25]:
bus_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35667 entries, 0 to 35666
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   From       35667 non-null  object
 1   To         35667 non-null  object
 2   Operator   35667 non-null  object
 3   Distance   35667 non-null  int64 
 4   Duration   35667 non-null  object
 5   Bus Type   35667 non-null  object
 6   Departure  35667 non-null  object
 7   Arrival    35667 non-null  object
dtypes: int64(1), object(7)
memory usage: 2.2+ MB


In [26]:
# no Null values thus far. But there may be other ways, in which the data is incomplete
bus_df.isnull().sum()

From         0
To           0
Operator     0
Distance     0
Duration     0
Bus Type     0
Departure    0
Arrival      0
dtype: int64

In [27]:

with open("data/bus_starting_output.txt", "w") as f:
  print(bus_df["From"].value_counts(dropna=False), file=f)

In [28]:

with open("data/bus_destination_output.txt", "w") as f:
  print(bus_df["To"].value_counts(dropna=False), file=f)

Findings:
+ All origins and destinations are properly capitalized, and most of them have correct formatting. 
+ No rows have missing city names for origin or destination
+ Some places have "bypass" in their names. The number of routes with those places may be statistically insignificant
    + But it is still worthwhile to find out how to treat them properly


In [29]:
with open("data/bus_operator_output.txt", "w") as f:
  print(bus_df["Operator"].value_counts(dropna=False), file=f)

For operators, it's more complicated:
+ Different operators have very similar names
    + i.e. sharma travels (nanded)-(chintamani) and sharma travels (nanded)-devgiri express-VT
    + Are they the parts of the same company? Or should they be treated as separate entities?
    + there are many such cases, so this is a more important problem than the "bypass" problem mentioned above.

In [30]:
with open("data/bus_distance_output.txt", "w") as f:
  print(bus_df["Distance"].value_counts(dropna=False, normalize=True), file=f)

Distance between cities measured in kilometers - pretty straightforward. Distances of hundreds of kilometers are common, but that's not surprising.

In [31]:
with open("data/travel_duration_output.txt", "w") as f:
  print(bus_df["Duration"].value_counts(dropna=False), file=f)

The original dataset claims that duration is in the format "hh:mm:ss", but that is wrong. The second number in the timestamp is never greater than 23, implying that it is the hour, and not the minute. Moreover, distances between cities are so long, that it is impossible to clear them in mere hours. Therefore, the more appropriate timestamp format would be "d:hh:mm".

Later, I will convert the "Duration" column into a proper timestamp recognized by pandas

In [32]:
with open("data/bus_type_output.txt", "w") as f:
  print(bus_df["Bus Type"].value_counts(dropna=False), file=f)

This is the most difficult column to clean. We must split it into several different columns, each containing the following information:
+ sleeper vs non-sleeper - represents the extent, to which this bus is designed for sleeping
+ brand of bus
+ seat configuration(2+2, 1+2, etc.) - represents # of seats per row
+ AC vs non-AC - presence of air conditioning
+ miscellaneous - descriptions that do not fit into any of the above categories

In [33]:
with open("data/departure_time_output.txt", "w") as f:
  print(bus_df["Departure"].value_counts(dropna=False), file=f)

In [34]:
with open("data/arrival_time_output.txt", "w") as f:
  print(bus_df["Arrival"].value_counts(dropna=False), file=f)

Both departure and arrival follow the same straightforward scheme. hh:mm:ss 12-hour format.
Must convert them to proper timestamp format

# Cleaning

I clean up data by changing format and splitting the "Bus Type" column into many smaller types.

In [35]:
import regex as re

# I cloned this dataframe, so that it's easier to trace it.
bus_df_v2 = bus_df.copy()
# Convert Bus Type column to lowercase to simplify detection

bt = bus_df_v2["Bus Type"].str.lower()

In [36]:


# this code will separate the many values inside the Bus Type column into something useful.
# 1. Detect explicit Non-AC
non_ac_pat = r"non a/c|non-ac|nonac|non ac"
has_non_ac = bt.str.contains(non_ac_pat)

# 2. Detect explicit AC (but not "non ac")
ac_pat = r"\ba/c\b|\bac\b"
has_ac = bt.str.contains(ac_pat)

# 3. Single 'has_ac' column:
#    - True  = explicitly AC and not Non-AC
#    - False = explicitly Non-AC and not AC
#    - NaN   = ambiguous or neither mentioned
bus_df_v2["has_ac"] = pd.Series(pd.NA, index=bus_df_v2.index, dtype="boolean")

bus_df_v2.loc[has_ac & ~has_non_ac, "has_ac"] = True      # definite AC
bus_df_v2.loc[has_non_ac, "has_ac"] = False     # definite Non-AC
# bus_df_v2.loc[has_ac & has_non_ac, "has_ac"] = pd.NA      
# weird strings like "Non A/c  A/c"
# rows where neither has_ac nor has_non_ac stay as NaN




In [37]:

# 1. Detect Semi-Sleeper first
semi_pat = r"semi[\s-]?sleeper"
is_semi_sleeper = bt.str.contains(semi_pat)

# 2. Any mention of "sleeper" at all
has_sleeper_word = bt.str.contains("sleeper")

# -------- Sleeper vs Semi-Sleeper (no overlap) --------
bus_df_v2["sleeper_category"] = pd.Series(pd.NA, index=bus_df_v2.index, dtype="object")


# 3. Separate columns:
#    - is_semi_sleeper: True/False (simple boolean)
#    - is_sleeper: True if sleeper but not semi-sleeper, NaN otherwise
bus_df_v2.loc[is_semi_sleeper, "sleeper_category"] = "Semi-Sleeper"
bus_df_v2.loc[has_sleeper_word & ~is_semi_sleeper, "sleeper_category"] = "Sleeper"
bus_df_v2.loc[~has_sleeper_word & ~is_semi_sleeper, "sleeper_category"] = "Other"

# if a bus is neither sleeper nor semi-sleeper, "sleeper_category" is "Other"

In [38]:
bus_df_v2["is_seater"]      = bt.str.contains("seater|seat")
bus_df_v2["is_multiaxle"]   = bt.str.contains("multi")
bus_df_v2["has_video"]      = bt.str.contains("video")
bus_df_v2["is_pushback"]    = bt.str.contains("push")
# there are many different seat formats, but majority of values obeys a similar formatting
seat_format_pat = r"(\d\s*[\+x]\s*\d)"
bus_df_v2["seat_format"] = bt.str.extract(seat_format_pat, expand=False)# use Regex to turn "brand" into a separate categorical variable. More convenient than creating booleans for each brand
bus_df_v2["brand"]       = bt.str.extract(r"(volvo|mercedes|isuzu|king long)", flags=re.IGNORECASE)

Afterwards, I save the snapshot of the dataframe to CSV, so that it's easier to retrace steps in case smth got lost.

In [39]:
bus_df_v2.to_csv("data/Pan-India_Bus_Routes_v2.csv")

In [40]:
bus_df_v2.head()

,From,To,Operator,Distance,Duration,Bus Type,Departure,Arrival,has_ac,sleeper_category,is_seater,is_multiaxle,has_video,is_pushback,seat_format,brand
0,Sattur,Chennai,PERINBA VILAS TRAVELS,539,0:9:0,"A/C, 35 Seat, 2+2 Semi Sleeper, Air Suspensio...",09:30:00 PM,06:30:00 AM,True,Semi-Sleeper,True,False,False,False,2+2,NaN
1,Sattur,Chennai,PERINBA VILAS TRAVELS,539,0:10:45,"Non A/C, 34 Seat, 2+1 Executive, Air Suspensio...",07:45:00 PM,06:30:00 AM,False,Other,True,False,False,False,2+1,NaN
2,Sattur,Chennai,KPN Travels,539,0:9:45,2+2 : 33 S.S NON A/C,07:45:00 PM,05:30:00 AM,False,Other,False,False,False,False,2+2,NaN
3,Sattur,Salem,Srs Travels (SRS Travels),304,0:5:45,"2+1, Sleeper, AC, Non-Video",09:45:00 PM,03:30:00 AM,True,Sleeper,False,False,True,False,2+1,NaN
4,Sattur,Coimbatore,India Travel Service,295,0:6:40,2+2 Hitech Air Bus Non/AC,10:20:00 PM,05:00:00 AM,True,Other,False,False,False,False,2+2,NaN


In [41]:
bus_df_v2["sleeper_category"].value_counts(dropna=False)

sleeper_category
Other           21467
Semi-Sleeper     8158
Sleeper          6042
Name: count, dtype: int64

For the overwhelming majority of bus trips, the brand of the bus is not specified. This will interfere with data significantly. Same goes for seat format. 

In [42]:
bus_df_v2["brand"].value_counts(dropna=False)

brand
NaN          26319
volvo         8493
mercedes       608
isuzu          246
king long        1
Name: count, dtype: int64

In [43]:
bus_df_v2["seat_format"].value_counts(dropna=False)

seat_format
NaN      27209
2+2       4833
2+1       1765
1+2       1529
2x1        167
1+1         73
2 x 1       30
2+3         29
2x2         26
2 +1         6
Name: count, dtype: int64

In [44]:
bus_df_v2[(bus_df_v2["is_seater"]==False) & (bus_df_v2["is_pushback"]==True)].head()

,From,To,Operator,Distance,Duration,Bus Type,Departure,Arrival,has_ac,sleeper_category,is_seater,is_multiaxle,has_video,is_pushback,seat_format,brand
1725,Bhramavar,Gadag,Sri Durgamba Travels,346,0:7:40,2+3SLIGHT PUSH BACK,09:50:00 PM,05:30:00 AM,<NA>,Other,False,False,False,True,2+3,NaN
1728,Bhramavar,Hubli,Sri Durgamba Travels,291,0:6:10,2+3SLIGHT PUSH BACK,09:50:00 PM,04:00:00 AM,<NA>,Other,False,False,False,True,2+3,NaN
2304,Chennai,Kovilpati,Subadhdra Travels,564,0:14:6,2+2 PushBack Non/Ac,07:00:00 PM,09:06:00 AM,True,Other,False,False,False,True,2+2,NaN
2314,Chennai,Kovilpati,Tippu Sultan Travels,564,0:14:6,Pushback,07:00:00 PM,09:06:00 AM,<NA>,Other,False,False,False,True,NaN,NaN
2325,Chennai,Thakkalai,Tippu Sultan Travels,324,0:8:6,Pushback,07:00:00 PM,03:06:00 AM,<NA>,Other,False,False,False,True,NaN,NaN


For A/C it is straightforward. We know for most bus trips whether that bus had AC or not. And for a slight majority of bus trips, the bus was air-conditioned.

In [45]:
bus_df_v2["has_ac"].value_counts(dropna=False)

has_ac
True     15362
False    13763
<NA>      6542
Name: count, dtype: Int64

Now, I will clean bus trip duration variable.

In [46]:
bus_df_v3 = bus_df_v2.copy()


# split into 3 new columns
split = bus_df_v3["Duration"].str.split(":", expand=True)
bus_df_v3["days"] = split[0].astype(int)
bus_df_v3["hours"] = split[1].astype(int)
bus_df_v3["minutes"] = split[2].astype(int)

# convert into timedelta
bus_df_v3["duration_td"] = (
    pd.to_timedelta(bus_df_v3["days"], unit="D") +
    pd.to_timedelta(bus_df_v3["hours"], unit="H") +
    pd.to_timedelta(bus_df_v3["minutes"], unit="m")
)


/tmp/ipykernel_44778/1307858945.py:13: FutureWarning: 'H' is deprecated and will be removed in a future version. Please use 'h' instead of 'H'.
  pd.to_timedelta(bus_df_v3["hours"], unit="H") +


As a result, the duration_td column conversion worked. And now, all the values in it are timedelta values, easily interpretable by humans and machine learning models alike.

The format is `d days, hh:mm:ss`.

In [47]:
bus_df_v3["duration_td"].head()

0   0 days 09:00:00
1   0 days 10:45:00
2   0 days 09:45:00
3   0 days 05:45:00
4   0 days 06:40:00
Name: duration_td, dtype: timedelta64[ns]

In [48]:
bus_df_v3.to_csv("data/Pan-India_Bus_Routes_v3.csv")

In [49]:
bus_df_v3.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35667 entries, 0 to 35666
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype          
---  ------            --------------  -----          
 0   From              35667 non-null  object         
 1   To                35667 non-null  object         
 2   Operator          35667 non-null  object         
 3   Distance          35667 non-null  int64          
 4   Duration          35667 non-null  object         
 5   Bus Type          35667 non-null  object         
 6   Departure         35667 non-null  object         
 7   Arrival           35667 non-null  object         
 8   has_ac            29125 non-null  boolean        
 9   sleeper_category  35667 non-null  object         
 10  is_seater         35667 non-null  bool           
 11  is_multiaxle      35667 non-null  bool           
 12  has_video         35667 non-null  bool           
 13  is_pushback       35667 non-null  bool           
 14  seat_f

Now, I am pruning unnecessary columns from the dataset, as they have already served their purpose in the cleaning. 

In [50]:
bus_df_v4 = bus_df_v3.copy()

In [51]:
bus_df_v4 = bus_df_v4.drop(["Duration", "Bus Type", "days", "hours", "minutes"], axis=1)
bus_df_v4.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35667 entries, 0 to 35666
Data columns (total 15 columns):
 #   Column            Non-Null Count  Dtype          
---  ------            --------------  -----          
 0   From              35667 non-null  object         
 1   To                35667 non-null  object         
 2   Operator          35667 non-null  object         
 3   Distance          35667 non-null  int64          
 4   Departure         35667 non-null  object         
 5   Arrival           35667 non-null  object         
 6   has_ac            29125 non-null  boolean        
 7   sleeper_category  35667 non-null  object         
 8   is_seater         35667 non-null  bool           
 9   is_multiaxle      35667 non-null  bool           
 10  has_video         35667 non-null  bool           
 11  is_pushback       35667 non-null  bool           
 12  seat_format       8458 non-null   object         
 13  brand             9348 non-null   object         
 14  durati

In [52]:
bus_df_v4.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35667 entries, 0 to 35666
Data columns (total 15 columns):
 #   Column            Non-Null Count  Dtype          
---  ------            --------------  -----          
 0   From              35667 non-null  object         
 1   To                35667 non-null  object         
 2   Operator          35667 non-null  object         
 3   Distance          35667 non-null  int64          
 4   Departure         35667 non-null  object         
 5   Arrival           35667 non-null  object         
 6   has_ac            29125 non-null  boolean        
 7   sleeper_category  35667 non-null  object         
 8   is_seater         35667 non-null  bool           
 9   is_multiaxle      35667 non-null  bool           
 10  has_video         35667 non-null  bool           
 11  is_pushback       35667 non-null  bool           
 12  seat_format       8458 non-null   object         
 13  brand             9348 non-null   object         
 14  durati

Just wanted to check that the "duration_td" variable has no formatting errors. It does not, thankfully.

In [53]:
with open("data/duration_td.txt", "w") as f:
  print(bus_df_v4["duration_td"].value_counts(dropna=False), file=f)

Renamed columns. A minor step, but it goes a long way to beauty and convenience of the dataframe.

In [54]:
column_rename_dictionary = {"has_ac":"Has A/C?", "sleeper_category":"Sleeper Category", "is_seater": "Is Seater?", "is_multiaxle": "Is Multi-Axle?", 
"has_video": "Has Video?", "is_pushback": "Pushback Seats", "seat_format": "Seat Format", "brand": "Bus Brand", "duration_td": "Duration"}

bus_df_v5 = bus_df_v4.rename(columns=column_rename_dictionary)

Saving this version of the dataframe to v5. Why not v4? Because while cloning the dataframe under a different variable name is significant for Python code, it is not so important as to warrant a separate .csv file. But the name of that file has to end with "v5", because it's necessary for understanding to which version of the dataframe it corresponds.

In [55]:
bus_df_v5.to_csv("data/Pan-India_Bus_Routes_v5.csv")